# Dayhoff Quickstart

A quick tour of Dayhoff: 
1. **Setup** Check that dependencies were installed correctly 
2. **Model loading** Load a small model
3. **Generate** sequences unconditionally 
4. **Zero-shot scoring** using l->r and r->l decoding 
5. **Hugging Face** dataset streaming

First time here? 

From `examples/`, run `./setup_cpu.sh` or `./setup_gpu.sh`, then pick the
kernel it creates. 

No GPU? No problem — the model cells skip themselves and everything else
still runs.

More detail lives in the [README](../README.md) and the
[preprint](https://aka.ms/dayhoff/preprint).

## Check Installation

A quick look at your setup. `RUN_MODEL` just tells the later cells whether they can load the
model (you need a GPU plus the `mamba-ssm` / `flash-attn` kernels).

In [5]:
import importlib.util
import platform
import torch

has_gpu = torch.cuda.is_available()
has_kernels = all(importlib.util.find_spec(m)
                  for m in ("mamba_ssm", "flash_attn", "causal_conv1d"))
RUN_MODEL = has_gpu and has_kernels

print("Python ", platform.python_version())
print("PyTorch", torch.__version__)
print("GPU    ", torch.cuda.get_device_name(0) if has_gpu else "none")
print("Model:", "ready to run on GPUs" if RUN_MODEL else "skipped (CPU / dataset-only)")

Python  3.12.11
PyTorch 2.8.0+cu128
GPU     NVIDIA RTX A6000
Model: ready to run on GPUs


## Load a model

We use the smallest checkpoint, `Dayhoff-170m-UR50`. Point `MODEL_NAME` at any model from the
README to try a bigger one (e.g. `Dayhoff-3b-GR-HM-c`).

In [6]:
MODEL_NAME = "Dayhoff-170m-UR50"
HF_ID = f"microsoft/{MODEL_NAME}"

model = tokenizer = None
if RUN_MODEL:
    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

    set_seed(0)
    model = AutoModelForCausalLM.from_pretrained(HF_ID).to("cuda:0").to(torch.bfloat16).eval()
    tokenizer = AutoTokenizer.from_pretrained(HF_ID, trust_remote_code=True)
    print("Loaded", HF_ID)
else:
    print("No GPU — skipping the model. Jump ahead to the dataset section.")

/home/salamdari/anaconda3/envs/bio-model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded microsoft/Dayhoff-170m-UR50


## Generate sequences

Same idea as `generate.py`: start from the `START` token, only let the model emit amino
acids, and decode what comes out.

In [ ]:
num_sequences = 2

if RUN_MODEL:
    from transformers import SuppressTokensLogitsProcessor
    from dayhoff.constants import constants

    aas, START, STOP = constants.CAN_AAS, constants.START, constants.STOP
    vocab = tokenizer.alphabet
    keep = {vocab.index(c) for c in aas} | {vocab.index(STOP)}
    block = [t for t in range(len(vocab)) if t not in keep]
    model.generation_config.eos_token_id = vocab.index(STOP)

    for i in range(num_sequences):
        ids = tokenizer(START, return_tensors="pt",
                        return_token_type_ids=False)["input_ids"].to(model.device)
        out = model.generate(ids, 
                             do_sample=True, 
                             max_new_tokens=512, 
                             temperature=1.0, 
                             min_p=0.0,
                             logits_processor=[SuppressTokensLogitsProcessor(block, device=model.device)])
        seq = tokenizer.batch_decode(out)[0].replace(START, "").replace(STOP, "")
        print(f">seq_{i}\n{seq}\n")

>seq_0
MVGFWTIRRIEFGANSLTFVCPGLVVYLDSTRALFFQ

>seq_1
MSSIVSKGHSQLPNDVGVHKRVHEEYVRRYPGFIVDGVREDHVSPTAFSLVGMTRRARAHQPA



## Score sequences

Forward and backward log-likelihoods, like `score.py`. Higher (less negative) means the model
likes the sequence more (this is what powers zero-shot mutation scoring).

In [ ]:
if RUN_MODEL:
    import pandas as pd
    from dayhoff.constants import constants
    
    START, STOP = constants.START, constants.STOP

    def score(seq):
        ids = tokenizer([START + seq + STOP, STOP + seq[::-1] + START],
                        return_tensors="pt", return_token_type_ids=False)["input_ids"].to(model.device)
        with torch.no_grad():
            fwd = -model(input_ids=ids[:1], labels=ids[:1]).loss.item()
            bwd = -model(input_ids=ids[1:], labels=ids[1:]).loss.item()
        return fwd, bwd

    seqs = {
        "seq_a": "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQ",
        "seq_b": "MGSSHHHHHHSSGLVPRGSHMASMTGGQQMGRGSEF",
    }
    rows = []
    for name, seq in seqs.items():
        fwd, bwd = score(seq)
        rows.append({"name": name, "len": len(seq), "fwd": fwd, "bwd": bwd, "avg": (fwd + bwd) / 2})
        # we use avg scores in the manuscript 

    display(pd.DataFrame(rows))


,name,len,fwd,bwd,avg
0,seq_a,66,-2.856896,-2.835224,-2.846060
1,seq_b,36,-1.489209,-2.578066,-2.033637


## Stream a dataset

Grab a few records from `microsoft/DayhoffDataset` without downloading the whole thing.
`streaming=True` only pulls what you actually iterate over. Works on any kernel.

In [12]:
from datasets import load_dataset

num_records = 3 

ds = load_dataset("microsoft/DayhoffDataset", name="uniref50", split="train", streaming=True)
for i, record in zip(range(num_records), ds):
    short = {k: (v[:60] + "…" if isinstance(v, str) and len(v) > 60 else v)
             for k, v in record.items()}
    print(f"record {i}: {short}")

record 0: {'index': 0, 'sequence': 'MEEITQIKKRLSQTVRLEGKEDLLSKKDSITNLKTEEHVSVKKMVISEPKPEKKEDIQLK…', 'accession': 'UniRef50_A0A5A9P0L4', 'description': 'peptidylprolyl isomerase n=1 Tax=Triplophysa tibetana TaxID=…'}
record 1: {'index': 1, 'sequence': 'MKAIAWLIVLTFLPEQVAWAVDYNLRGALHGAVAPLVSAATVATDGSVLQKTQKDEAFDR…', 'accession': 'UniRef50_A0A410P257', 'description': 'Glycogen synthase n=2 Tax=Velamenicoccus archaeovorus TaxID=…'}
record 2: {'index': 2, 'sequence': 'MEILGRNLPRILGNLVKTIKTAPVRVVARRGARTLTQKEFGKYLGSDAFKAVARRNIPFM…', 'accession': 'UniRef50_A0A8J3NBY6', 'description': 'Uncharacterized protein n=2 Tax=Actinocatenispora rupis TaxI…'}


## A few tips

- **Are model cells skipping?** You're on the CPU path. Run `./setup_gpu.sh` and switch to the
  `dayhoff-gpu` kernel in the notebook to generate and score.
- **First run is slow** — weights and data download once, then they're cached.
- **Out of memory?** Stick with the 170M model and a small `max_new_tokens`.

The README has the full install and Docker details.